# 10 — Validate the overall first-culture selection

## Goal

The conventional susceptibility population must contain **one earliest
eligible culture per source release, patient, cohort and reporting
organism**, irrespective of resistance phenotype. This notebook also
reconstructs the **retired phenotype-first method** to show why it
could contribute multiple records per patient–organism group.

The historical comparison is derived directly from
`int_reporting_culture_organism`; it does **not** require the retired
phenotype-first dbt tables. The published susceptibility mart uses
`int_overall_first_reporting_isolate` and its own AST reconciliation;
`mart_mdro` uses a separate patient-level evidence policy.

Prerequisite: successfully build
`int_overall_first_reporting_susceptibility` and its upstream models.
Run these cells against the personal Databricks `workspace.micro_dev`
schema. ARMD provides shifted culture times but no verified isolate ID;
this is a first-*culture* rule defined for this research project.


## 1. Reconstruct the historical phenotype-first population

The retired method selected one earliest culture **within each
phenotype**, after assigning source organism records to phenotype
categories. The SQL reconstructs that selection with `row_number()`
using the original timestamp and key ordering. A patient with both a
base and resistant phenotype can contribute more than one record.

The output shows unique patient–organism groups alongside historical
phenotype-first contributions. The difference is not an extra count of
unique patients.


In [ ]:
%sql
with phenotype_ranked as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,
        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype
            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as phenotype_rank
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
phenotype_first as (
    select *
    from phenotype_ranked
    where phenotype_rank = 1
),
patient_groups as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        count(*) as phenotype_first_records
    from phenotype_first
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
)
select
    cohort_id,
    organism_reporting_group,
    count(*) as unique_patient_organism_groups,
    sum(phenotype_first_records) as historical_phenotype_first_records,
    sum(phenotype_first_records - 1) as extra_phenotype_contributions,
    sum(case
        when phenotype_first_records > 1 then 1 else 0
    end) as patients_in_multiple_phenotypes
from patient_groups
group by cohort_id, organism_reporting_group
order by patients_in_multiple_phenotypes desc,
         cohort_id, organism_reporting_group;


## 2. Compare historical and corrected populations

One row is expected per reporting organism/cohort/source release. The
`overall_first_records` count must equal the number of unique patients
in the historically reconstructed population, whereas the historical
phenotype-first count may be greater. **Every `population_check` must
be `pass`.**


In [ ]:
%sql
with phenotype_ranked as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,
        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype
            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as phenotype_rank
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
phenotype_summary as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        count(*) as historical_phenotype_first_records,
        count(distinct patient_key) as unique_patients
    from phenotype_ranked
    where phenotype_rank = 1
    group by source_version, cohort_id, organism_reporting_group
),
overall_summary as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        count(*) as overall_first_records
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by source_version, cohort_id, organism_reporting_group
)
select
    coalesce(p.source_version, o.source_version) as source_version,
    coalesce(p.cohort_id, o.cohort_id) as cohort_id,
    coalesce(p.organism_reporting_group, o.organism_reporting_group)
        as organism_reporting_group,
    p.unique_patients,
    p.historical_phenotype_first_records,
    o.overall_first_records,
    p.historical_phenotype_first_records - o.overall_first_records
        as extra_phenotype_contributions,
    case
        when p.unique_patients = o.overall_first_records
         and p.historical_phenotype_first_records >= o.overall_first_records
            then 'pass'
        else 'mismatch'
    end as population_check
from phenotype_summary p
full outer join overall_summary o
    on p.source_version = o.source_version
   and p.cohort_id = o.cohort_id
   and p.organism_reporting_group = o.organism_reporting_group
order by
    case
        when p.unique_patients = o.overall_first_records
         and p.historical_phenotype_first_records >= o.overall_first_records
            then 1 else 0
    end,
    extra_phenotype_contributions desc,
    cohort_id, organism_reporting_group;


## 3. Verify the overall first-culture grain

Each `source_version × cohort_id × patient_key × organism_reporting_group`
must appear exactly once. The surrogate key must be unique and the
selected record must retain at least one source culture-organism key.
**Expect zeros for all error-count columns.**


In [ ]:
%sql
with duplicate_groups as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        count(*) as n
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
    having count(*) > 1
)
select
    count(*) as overall_records,
    count(distinct overall_reporting_isolate_key) as unique_overall_keys,
    count(*) - count(distinct overall_reporting_isolate_key) as duplicate_keys,
    (select count(*) from duplicate_groups) as duplicate_groups,
    sum(case when source_culture_organism_keys is null
              or size(source_culture_organism_keys) = 0
             then 1 else 0 end) as records_missing_source_keys
from workspace.micro_dev.int_overall_first_reporting_isolate;


### Independently reconstruct the earliest culture

Rebuild the ranking from distinct culture-level records **without**
consulting antibiotic availability or susceptibility interpretation.
This guards against inadvertently selecting a later, tested culture.
**Expect `selection_mismatches = 0`.**


In [ ]:
%sql
with observed_cultures as (
    select distinct
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key,
        culture_time_shifted
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
independent_rank as (
    select
        *,
        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group
            order by culture_time_shifted, culture_key
        ) as rn
    from observed_cultures
),
expected as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from independent_rank
    where rn = 1
),
actual as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        culture_key
    from workspace.micro_dev.int_overall_first_reporting_isolate
),
comparison as (
    select
        e.culture_key as expected_culture,
        a.culture_key as selected_culture
    from expected e
    full outer join actual a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group = a.organism_reporting_group
)
select
    count(*) as compared_groups,
    sum(case when expected_culture = selected_culture
             then 0 else 1 end) as selection_mismatches
from comparison;


### Check source-label consolidation on the selected culture

The overall first-culture record must include all source organism keys
across phenotype categories on that culture. This comparison rebuilds
the array directly from the surviving culture-level model. The two
mismatch counts must be zero.


In [ ]:
%sql
with expected_source_records as (
    select
        f.overall_reporting_isolate_key,
        array_sort(array_distinct(flatten(
            collect_list(r.source_culture_organism_keys)
        ))) as expected_source_keys,
        count(distinct r.reporting_culture_organism_key)
            as expected_phenotype_records_on_culture
    from workspace.micro_dev.int_overall_first_reporting_isolate f
    inner join workspace.micro_dev.int_reporting_culture_organism r
        on f.source_version = r.source_version
       and f.cohort_id = r.cohort_id
       and f.patient_key = r.patient_key
       and f.organism_reporting_group = r.organism_reporting_group
       and f.culture_key = r.culture_key
    group by f.overall_reporting_isolate_key
)
select
    count(*) as selected_cultures_compared,
    sum(case
        when e.overall_reporting_isolate_key is null
          or f.source_culture_organism_keys <> e.expected_source_keys
            then 1 else 0
    end) as source_key_array_mismatches,
    sum(case
        when e.overall_reporting_isolate_key is null
          or f.phenotype_records_on_culture
             <> e.expected_phenotype_records_on_culture
            then 1 else 0
    end) as phenotype_record_count_mismatches
from workspace.micro_dev.int_overall_first_reporting_isolate f
left join expected_source_records e
    on f.overall_reporting_isolate_key = e.overall_reporting_isolate_key;


## 4. Validate overall-culture AST reconciliation

`int_overall_first_reporting_susceptibility` combines every source
organism record on the selected culture. It preserves missing and
conflicting endpoints rather than choosing an arbitrary source label.
Surrogate keys must be unique. Resolved endpoints must contain a
result; unresolved endpoints must not.


In [ ]:
%sql
select
    count(*) as endpoint_rows,
    count(distinct overall_reporting_susceptibility_key) as unique_keys,
    sum(case when resolution_status = 'resolved'
              and susceptibility is null
             then 1 else 0 end) as resolved_with_null,
    sum(case when resolution_status <> 'resolved'
              and susceptibility is not null
             then 1 else 0 end) as unresolved_with_result
from workspace.micro_dev.int_overall_first_reporting_susceptibility;


### Audit resolved, missing and conflicting endpoints

Inspect the complete resolution/reconciliation breakdown, including
endpoints for which more than one source organism label contributed.
This is evidence for the audit trail, not a filter for selecting which
culture enters the overall population.


In [ ]:
%sql
select
    resolution_status,
    reconciliation_reason,
    count(*) as endpoint_rows,
    sum(case when source_organisms_with_endpoint > 1
             then 1 else 0 end) as multi_source_endpoints
from workspace.micro_dev.int_overall_first_reporting_susceptibility
group by resolution_status, reconciliation_reason
order by resolution_status, reconciliation_reason;


## Conclusion

The historical phenotype-first selection is reproducible **as a
diagnostic query** without two additional production dbt tables.
The current susceptibility population selects the earliest culture
**across phenotypes** and reconciles all source labels on that culture.
Notebook 11 validates the final susceptibility mart. Notebook 12
separately validates the all-observation, patient-level MDRO policy.

Rerun all cells after this cleanup; outputs from the old notebook have
been cleared rather than copied as evidence of the revised queries.
